In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm
from datetime import datetime, timedelta

pd.set_option("mode.chained_assignment", None)


import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

plt.rcParams["font.family"] = "Arial"

from additives_microbiota.plotting import process_string
from additives_microbiota.constants import ADDITIVE_VARS, MIXTURES
from additives_microbiota.mb_utils import transform_counts_data, preprocess_picrust_data

### Loading Meta data

In [2]:
meta = pd.read_csv("../../data/metadata_files/meta_zscores.csv", index_col=0)

### Loading MB data

In [3]:
picrust_data_type = "pathway"  # pathway | ko | ec

if picrust_data_type == "pathway":
    picrust_data = pd.read_csv(
        "~/WORK/qiime2_2024/qiime_processed_files/exported_rep_seqs/pathways_out/path_abun_unstrat_descrip.tsv",
        sep="\t",
    )
    desc_dict = picrust_data.set_index("pathway")["description"].to_dict()
    picrust_data = picrust_data.drop(columns=["description"]).set_index("pathway").T
elif picrust_data_type == "ko":
    picrust_data = pd.read_csv(
        "~/WORK/qiime2_2024/qiime_processed_files/exported_rep_seqs/KO_metagenome_out/pred_metagenome_unstrat_descrip.tsv",
        sep="\t",
    )

    desc_dict = picrust_data.set_index("function")["description"].to_dict()
    picrust_data = picrust_data.drop(columns=["description"]).set_index("function").T
elif picrust_data_type == "ec":
    picrust_data = pd.read_csv(
        "~/WORK/qiime2_2024/qiime_processed_files/exported_rep_seqs/EC_metagenome_out/pred_metagenome_unstrat_descrip.tsv",
        sep="\t",
    )

    desc_dict = picrust_data.set_index("function")["description"].to_dict()
    picrust_data = picrust_data.drop(columns=["description"]).set_index("function").T
else:
    raise ValueError(f"Invalid picrust_data_type: {picrust_data_type}")

picrust_data.head()

In [ ]:
data_transformed_filt, id_dict, rev_id_dict, prevalence_filt = preprocess_picrust_data(
    picrust_data,
    data_type=picrust_data_type,
    method="log1p",
    min_prev=0.05,
    min_var=1e-8,
)

data_transformed_meta = meta.merge(
    data_transformed_filt, left_on="sample-id", right_index=True
)

print(data_transformed_meta.shape)
data_transformed_filt.head()

In [6]:
def run_association_models(
    data_transformed_filt,
    data_transformed_meta,
    base_formula,
    desc_dict,
    id_dict,
    food_var,
    alpha=0.05,
):
    """
    Run OLS models for each pathway feature, extract FOOD variable effect and p-value,
    and compute Benjamini-Hochberg FDR correction.

    Returns:
        results_df: DataFrame indexed by feature with beta, se, pval, qval_bh, signif_fdr05
    """
    results = []
    for feat in data_transformed_filt.columns:
        formula = base_formula.format(FOOD=food_var, MB_FEATURE=feat)
        m = smf.ols(formula, data=data_transformed_meta).fit(cov_type="HC3")

        # Extract stats for the FOOD term
        if food_var in m.params.index:
            results.append(
                {
                    "feature": feat,
                    "beta": float(m.params[food_var]),
                    "se": float(m.bse[food_var]),
                    "pval": float(m.pvalues[food_var]),
                    "n": int(m.nobs),
                }
            )

    results_df = pd.DataFrame(results).set_index("feature")

    # Benjamini-Hochberg FDR
    results_df = results_df.sort_values("pval")
    m_val = results_df.shape[0]
    ranks = np.arange(1, m_val + 1)

    results_df["qval_bh"] = (results_df["pval"] * m_val / ranks).clip(upper=1.0)
    # Enforce monotonicity of BH q-values
    results_df["qval_bh"] = results_df["qval_bh"][::-1].cummin()[::-1]

    # Flag significant pathways
    results_df["signif_fdr05"] = results_df["qval_bh"] < alpha

    # Add description
    # results_df["description"] = results_df.index.map(
    #     lambda x: desc_dict[[k for k, v in id_dict.items() if v == x][0]]
    # )
    results_df["description"] = results_df.index.map(
        lambda x: desc_dict.get(rev_id_dict.get(x), None)
    )

    results_df["pathway_id"] = results_df.index.map(
        lambda x: [k for k, v in id_dict.items() if v == x][0]
    )

    return results_df

In [7]:
base_formula = (
    "{MB_FEATURE} ~ urbanity + language + {FOOD} + "
    "C(gender) + age + bmi + C(antibiotics_treatment) + "
    "C(smoking, Treatment(reference='non-smoker')) + "
    "defecate_quantity_per_day + C(batch)"
)

In [8]:
# from tqdm.notebook import tqdm

ignore_vars = ["other_additives"]
all_results = []

# for food_var in tqdm(ADDITIVE_VARS, desc="Testing food vars"):
for food_var in ADDITIVE_VARS + MIXTURES:
    if food_var in ignore_vars:
        continue
    results_df = run_association_models(
        data_transformed_filt,
        data_transformed_meta,
        base_formula,
        desc_dict,
        id_dict,
        food_var,
        alpha=0.05,
    )
    if not results_df.empty:
        results_df = results_df.copy()
        results_df["food"] = food_var
        all_results.append(results_df)

if all_results:
    final_results_df = pd.concat(all_results)
    # print(final_results_df["description"].values)
    final_results_df
else:
    print("No significant results found.")
    pd.DataFrame()

In [9]:
final_results_df = pd.concat(all_results).reset_index()  # index is feature
final_results_df = final_results_df.rename(columns={"index": "feature"})

final_results_df = final_results_df.sort_values("pval")
m = final_results_df.shape[0]
ranks = np.arange(1, m + 1)

final_results_df["qval_bh_global"] = (final_results_df["pval"] * m / ranks).clip(
    upper=1.0
)
final_results_df["qval_bh_global"] = final_results_df["qval_bh_global"][::-1].cummin()[
    ::-1
]
final_results_df["signif_fdr05_global"] = final_results_df["qval_bh_global"] < 0.05

final_hits = final_results_df[final_results_df["signif_fdr05_global"]]

In [10]:
final_hits.sort_values(["food", "qval_bh_global"]).to_csv(
    f"results/ols_associations_{picrust_data_type}.csv"
)

In [11]:
final_hits.sort_values(["food", "qval_bh_global"])

,feature,beta,se,pval,n,qval_bh,signif_fdr05,description,pathway_id,food,qval_bh_global,signif_fdr05_global
3008,pathway_187,-0.759281,0.167621,0.000006,940,0.001409,True,ethylmalonyl-CoA pathway,PWY-5741,Mixture3,0.012679,True
3009,pathway_89,1.079971,0.241110,0.000007,940,0.001409,True,TCA cycle IV (2-oxoglutarate decarboxylase),P105-PWY,Mixture3,0.012679,True
3010,pathway_246,1.045182,0.256549,0.000046,940,0.004687,True,"superpathway of 2,3-butanediol biosynthesis",PWY-6396,Mixture3,0.026064,True
3011,pathway_63,0.437780,0.109340,0.000062,940,0.004687,True,superpathway of hexitol degradation (bacteria),HEXITOLDEGSUPER-PWY,Mixture3,0.026363,True
3012,pathway_57,1.103367,0.275577,0.000062,940,0.004687,True,"superpathway of glycolysis, pyruvate dehydroge...",GLYCOLYSIS-TCA-GLYOX-BYPASS,Mixture3,0.026363,True
3013,pathway_387,1.157875,0.299426,0.000110,940,0.006044,True,superpathway of glyoxylate bypass and TCA,TCA-GLYOX-BYPASS,Mixture3,0.035195,True
3014,pathway_17,2.299928,0.601539,0.000132,940,0.006044,True,catechol degradation to &beta;-ketoadipate,CATECHOL-ORTHO-CLEAVAGE-PWY,Mixture3,0.035195,True
3015,pathway_121,2.328934,0.613793,0.000148,940,0.006044,True,protocatechuate degradation II (ortho-cleavage...,PROTOCATECHUATE-ORTHO-CLEAVAGE-PWY,Mixture3,0.035195,True
3016,pathway_169,2.296392,0.610841,0.000170,940,0.006044,True,aromatic compounds degradation via &beta;-keto...,PWY-5431,Mixture3,0.035195,True
3017,pathway_165,2.296392,0.610841,0.000170,940,0.006044,True,catechol degradation III (ortho-cleavage pathway),PWY-5417,Mixture3,0.035195,True


In [21]:
final_hits[
    ["feature", "food", "description", "pathway_id", "beta", "qval_bh_global", "pval"]
].set_index("pathway_id").sort_values(["food", "qval_bh_global"], ascending=False)

,feature,food,description,beta,qval_bh_global,pval
pathway_id,,,,,,
P101-PWY,pathway_88,sweeteners_glazing_agents,ectoine biosynthesis,-0.049227,0.046092,0.000259
PWY-6728,pathway_274,sweeteners_glazing_agents,methylaspartate cycle,-0.111710,0.035195,0.000141
PWY-7234,pathway_316,sweeteners_glazing_agents,inosine-5'-phosphate biosynthesis III,0.042116,0.026064,0.000039
PWY-6396,pathway_246,sweeteners_glazing_agents,"superpathway of 2,3-butanediol biosynthesis",0.093843,0.026064,0.000040
HEXITOLDEGSUPER-PWY,pathway_63,sweeteners_glazing_agents,superpathway of hexitol degradation (bacteria),0.041573,0.020133,0.000018
PWY-6397,pathway_247,ph_regulators_anti_caking_leavening_agents,mycolyl-arabinogalactan-peptidoglycan complex ...,-0.043498,0.035195,0.000166
LACTOSECAT-PWY,pathway_71,colorants,lactose and galactose degradation I,0.210156,0.031076,0.000083
PWY-6728,pathway_274,Mixture3,methylaspartate cycle,-1.252518,0.046160,0.000273
GLYOXYLATE-BYPASS,pathway_58,Mixture3,glyoxylate cycle,1.020751,0.044654,0.000238


In [24]:
print(
    final_hits[final_hits["food"] == "sweeteners_glazing_agents"]["pathway_id"].values
)

print(
    final_hits[final_hits["food"] == "sweeteners_glazing_agents"]["description"].values
)

['HEXITOLDEGSUPER-PWY' 'PWY-7234' 'PWY-6396' 'PWY-6728' 'P101-PWY']
['superpathway of hexitol degradation (bacteria)'
 "inosine-5'-phosphate biosynthesis III"
 'superpathway of 2,3-butanediol biosynthesis' 'methylaspartate cycle'
 'ectoine biosynthesis']


In [27]:
data_transformed_meta[
    [
        "daily_HEI",
        "shannon_entropy",
        "age",
        "bmi",
    ]
    + final_hits["feature"].unique().tolist()
].corr()["shannon_entropy"].sort_values(ascending=False)

shannon_entropy    1.000000
pathway_274        0.313197
daily_HEI          0.267515
pathway_57         0.258145
pathway_387        0.254713
pathway_89         0.232912
pathway_58         0.220013
pathway_88         0.204209
age                0.190963
pathway_187        0.164484
pathway_17         0.118028
pathway_169        0.117386
pathway_165        0.117386
pathway_247        0.053223
pathway_121        0.038252
pathway_246       -0.003245
pathway_71        -0.095603
bmi               -0.097833
pathway_316       -0.145504
pathway_63        -0.173612
pathway_110       -0.237726
Name: shannon_entropy, dtype: float64